In [1]:
from __future__ import annotations

import argparse
import importlib.util
import logging
import sys
import time
import types
from pathlib import Path
from tqdm import tqdm

from src.utils import pmf_utils
import pandas as pd
import matplotlib.pyplot as plt
import importlib, src.ddm.ddm
importlib.reload(src.ddm.ddm)

import numpy as np
import torch
import pickle

from config import dir_config

from src.ddm.utils import build_stimulus, get_job, build_grid, prepare_data, load_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [2]:
# change directory to project root
import os
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

from scripts.ddm.ddm_fitting import DDMModel

In [3]:
processed_dir = Path(dir_config.data.processed)
# ddm_dir = processed_dir / 'ddm'
ddm_dir = processed_dir / 'ddm_5a9ce9d'

session_metadata = pd.read_csv(processed_dir / "sessions_metadata.csv")
behavior_df = pd.read_csv(ddm_dir / "behavior_data.csv")

In [ ]:
session_ids = session_metadata["session_id"].tolist()
grid = build_grid(behavior_df)
job_lookup = {job_id: get_job(grid, job_id) for job_id in range(len(grid))}
stem_to_job_id = {
    (
        job["session_id"],
        job["prior_block"],
        job["enable_leak"],
        job["enable_time_constant"],
    ): job_id
    for job_id, job in job_lookup.items()
}

missing_job_ids = set()
for sub_dir in ddm_dir.iterdir():

    if not sub_dir.is_dir():
        continue

    # ----------------------------
    # parse variant from folder name
    # ----------------------------
    leak = "leak-1" in sub_dir.name
    tc = "tc-1" in sub_dir.name

    expected = {
        (sid, block, leak, tc)
        for sid in session_ids
        for block in [0, 1]
    }

    found = set()

    for model in sub_dir.rglob("*.pkl"):

        try:
            session = model.stem.split("_prior_block_")[0]
            block = int(model.stem.split("_prior_block_")[1])

            found.add((session, block, leak, tc))

        except Exception as e:
            print(f"[PARSE ERROR] {model}: {e}")

    missing = expected - found
    extra = found - expected

    print(
        f"\n=== {sub_dir.name} ===\n"
        f"Total files: {len(list(sub_dir.rglob('*.pkl')))}\t"
        f"Expected: {len(expected)}\t"
        f"Missing: {len(missing)}\t"
        f"Extra: {len(extra)}"
    )

    # ----------------------------
    # missing reporting
    # ----------------------------
    if missing:
        print("\nMISSING DETAILS:")
        for m in sorted(missing):
            job_id = stem_to_job_id.get(m)

            print(f"{m} → job_id={job_id}")

            if job_id is not None:
                missing_job_ids.add(job_id)

print("\n=== FITTING SUMMARY ===")

leaks = [0, 1]
tcs = [0, 1]
failed = {}
for leak in leaks:
    for tc in tcs:
        model_type = f"leak-{leak}_tc-{tc}"
        pkl_files = list((ddm_dir / model_type).glob("*.pkl"))
        failed = list((ddm_dir / model_type).glob("*.FAILED.json"))
        if failed:
            print(f"Model {model_type} has {len(failed)} failed fits. Missing job IDs:")
            for f in failed:
                try:
                    job_id = int(f.stem.split(".FAILED")[0])
                    print(f"  {f.name} → job_id={job_id}")
                except Exception as e:
                    print(f"  [PARSE ERROR] {f.name}: {e}")
        else:
            print(f"Model {model_type} has no failed fits.")

print("\n=== SUMMARY ===")
print("Missing job IDs:", sorted(missing_job_ids))


=== leak-0_tc-0 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== leak-1_tc-0 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== leak-0_tc-1 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== leak-1_tc-1 ===
Total files: 90	Expected: 90	Missing: 0	Extra: 0

=== FITTING SUMMARY ===
Model leak-0_tc-0 has no failed fits.
Model leak-0_tc-1 has no failed fits.
Model leak-1_tc-0 has no failed fits.
Model leak-1_tc-1 has no failed fits.

=== SUMMARY ===
Missing job IDs: []


# Session Wise Model Fits

In [9]:
# calculate cumulative likelihood for all model types
cumulative_likelihoods = {}

# calculate aic for all model types
aic_values = {}
bic_values = {}

leaks = [0, 1]
tcs = [0, 1]
PARAM_COUNTS = {
    "leak-0_tc-0": 5+2,  # ndt, a, z, drift_gain, drift_offset, sz, sdr
    "leak-0_tc-1": 6+2,  # + time_constant
    "leak-1_tc-0": 6+2,  # + leak_rate
    "leak-1_tc-1": 7+2,  # + both
}

for leak in leaks:
    for tc in tcs:
        model_type = f"leak-{leak}_tc-{tc}"
        model_path = ddm_dir / model_type

        pkl_files = list(model_path.glob("*.pkl"))

        total_nll = 0
        for pkl_file in tqdm(pkl_files, desc=f"Loading {model_type} models", unit="file"):
            try:
                model = load_model(pkl_file)  # just to check if it loads without error
                total_nll += model['results']['likelihood']
            except Exception as e:
                print(f"Error loading {pkl_file}: {e}")

        k = PARAM_COUNTS[model_type]
        cumulative_likelihoods[model_type] = np.round(total_nll)
        aic_values[model_type] = 2 * k + 2 * np.round(total_nll)  # since NLL is positive
        bic_values[model_type] = np.round(k * np.log(len(behavior_df)) + 2 * np.round(total_nll))  # BIC calculation


# capture key with highest likelihood (lowest NLL) and lowest AIC
best_likelihood_model = min(cumulative_likelihoods, key=cumulative_likelihoods.get)
best_aic_model = min(aic_values, key=aic_values.get)
best_bic_model = min(bic_values, key=bic_values.get)

if best_likelihood_model == best_aic_model == best_bic_model:
    best_model = best_likelihood_model
    print(f"Best model by both likelihood and AIC: {best_model}")
else:
    print(f"Best model by likelihood: {best_likelihood_model}")
    print(f"Best model by AIC: {best_aic_model}")
    print(f"Best model by BIC: {best_bic_model}")

Loading leak-1_tc-1 models: 100%|██████████| 90/90 [00:58<00:00,  1.55file/s]

Best model by both likelihood and AIC: leak-1_tc-1


In [10]:
print("=== MODEL COMPARISON ===")
print(f"{'Model':<15} {'Cumulative NLL':<20} {'AIC':<10} {'BIC':<10}")
for model in cumulative_likelihoods.keys():
    print(f"{model:<15} {cumulative_likelihoods[model]:<20} {aic_values[model]:<10} {bic_values[model]:<10}")


=== MODEL COMPARISON ===
Model           Cumulative NLL       AIC        BIC       
leak-0_tc-0     7548.0               15110.0    15171.0   
leak-0_tc-1     11241.0              22498.0    22568.0   
leak-1_tc-0     8736.0               17488.0    17558.0   
leak-1_tc-1     7395.0               14808.0    14887.0   
